Task 4.1 - Code of the CNN

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.optim as optim
from torch.utils.data import DataLoader

import torchvision.datasets as datasets
import torchvision.transforms as transforms

from torch.nn.functional import conv2d, max_pool2d, cross_entropy, dropout

plt.rc("figure", dpi=100)

batch_size = 100

# transform images into normalized tensors
train_transform = transforms.Compose([
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.MNIST(
    "./",
    download=True,
    train=True,
    transform=train_transform,
)

test_dataset = datasets.MNIST(
    "./",
    download=True,
    train=False,
    transform=test_transform,
)

train_dataloader = DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=1,
    pin_memory=True,
)

test_dataloader = DataLoader(
    dataset=test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=1,
    pin_memory=True,
)

def init_weights(shape):
    if len(shape) == 2:
        fan_in = shape[0]
    else:
        fan_in = np.prod(shape[1:])

    std = np.sqrt(2.0 / fan_in)
    w = torch.randn(shape) * std
    w.requires_grad = True

    return w


def rectify(x):
    # Rectified Linear Unit (ReLU)
    return torch.max(torch.zeros_like(x), x)


class RMSprop(optim.Optimizer):
    """
    This is a reduced version of the PyTorch internal RMSprop optimizer
    It serves here as an example
    """
    def __init__(self, params, lr=1e-3, alpha=0.5, eps=1e-8):
        defaults = dict(lr=lr, alpha=alpha, eps=eps)
        super(RMSprop, self).__init__(params, defaults)

    def step(self):
        for group in self.param_groups:
            for p in group['params']:
                grad = p.grad.data
                state = self.state[p]

                # state initialization
                if len(state) == 0:
                    state['square_avg'] = torch.zeros_like(p.data)

                square_avg = state['square_avg']
                alpha = group['alpha']

                # update running averages
                square_avg.mul_(alpha).addcmul_(grad, grad, value=1 - alpha)
                avg = square_avg.sqrt().add_(group['eps'])

                # gradient update
                p.data.addcdiv_(grad, avg, value=-group['lr'])


# define the neural network
def model(x, w_c1, w_c2, w_c3, w_h2, w_o, training=True, p_drop_input=0.2, p_drop_hidden=0.5):
    h1 = rectify(conv2d(x, w_c1))
    h1 = max_pool2d(h1, (2, 2))
    h1 = dropout(h1, p=p_drop_input, training=training)

    h2 = rectify(conv2d(h1, w_c2))
    h2 = max_pool2d(h2, (2, 2))
    h2 = dropout(h2, p=p_drop_input, training=training)

    h3 = rectify(conv2d(h2, w_c3))
    h3 = dropout(h3, p=p_drop_hidden, training=training)

    h3 = torch.reshape(h3, (h3.shape[0], -1))

    h4 = rectify(h3 @ w_h2)
    h4 = dropout(h4, p=p_drop_hidden, training=training)

    pre_softmax = h4 @ w_o

    return pre_softmax

def main():
    # initialize weights

    # input shape is (B, 784)
    w_c1 = init_weights((32, 1, 5, 5))
    w_c2 = init_weights((64, 32, 5, 5))
    w_c3 = init_weights((128, 64, 3, 3))

    w_h2 = init_weights((512, 625))
    w_o = init_weights((625, 10))
    # output shape is (B, 10)

    optimizer = RMSprop(params=[w_c1, w_c2, w_c3, w_h2, w_o], lr=1e-4, alpha=0.99)
    n_epochs = 100

    train_loss = []
    test_loss = []

    # put this into a training loop over 100 epochs
    for epoch in range(n_epochs + 1):
        train_loss_this_epoch = []
        for idx, batch in enumerate(train_dataloader):
            x, y = batch

            # our model requires flattened input
            x = x.reshape(-1, 1, 28, 28)
            # feed input through model
            noise_py_x = model(x, w_c1, w_c2, w_c3, w_h2, w_o, training=True)

            # reset the gradient
            optimizer.zero_grad()

            # the cross-entropy loss function already contains the softmax
            loss = cross_entropy(noise_py_x, y, reduction="mean")

            train_loss_this_epoch.append(loss.detach().item())

            # compute the gradient
            loss.backward()
            # update weights
            optimizer.step()

        train_loss.append(np.mean(train_loss_this_epoch))

        # test periodically
        if epoch % 10 == 0:
            print(f"Epoch: {epoch}")
            print(f"Mean Train Loss: {train_loss[-1]:.2e}")
            test_loss_this_epoch = []

            # no need to compute gradients for validation
            correct = 0
            total = 0
            with torch.no_grad():
                for idx, batch in enumerate(test_dataloader):
                    x, y = batch
                    x = x.reshape(-1, 1, 28, 28)
                    noise_py_x = model(x, w_c1, w_c2, w_c3, w_h2, w_o, training=False)

                    loss = cross_entropy(noise_py_x, y, reduction="mean")
                    test_loss_this_epoch.append(loss.detach().item())
                    pred = torch.argmax(noise_py_x, dim=1)

                    correct += (pred == y).sum().item()
                    total += y.size(0)

            test_loss.append(np.mean(test_loss_this_epoch))
            accuracy = correct / total
            test_error = 1 - accuracy

            print(f"Mean Test Loss: {test_loss[-1]:.2e}")
            print(f"Test Accuracy: {100 * accuracy:.2f}%")
            print(f"Test Error:    {100 * test_error:.2f}%")
            print(f"Mean Test Loss:  {test_loss[-1]:.2e}")

    plt.plot(np.arange(n_epochs + 1), train_loss, label="Train")
    plt.plot(np.arange(1, n_epochs + 2, 10), test_loss, label="Test")
    plt.title("Train and Test Loss over Training")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()

    # plot one image with convolution
    with torch.no_grad():
        x, y = next(iter(test_dataloader))
        image = x[0:1]
        conv_output = conv2d(image, w_c1)
        fig, ax = plt.subplots(2, 4, figsize=(10, 5))

        ax[0, 0].imshow(image[0, 0].numpy(), cmap="gray")
        ax[0, 0].set_title("Input")

        for i in range(3):
            ax[0, i + 1].imshow(conv_output[0, i].numpy(), cmap="gray")
            ax[0, i + 1].set_title(f"Feature map {i}")

            ax[1, i + 1].imshow(w_c1[i, 0].detach().numpy(), cmap="gray")
            ax[1, i + 1].set_title( f"Filter {i}")

        plt.tight_layout()
        plt.show()

if __name__ == "__main__":
    main()
